# FASTQ to ClinCNV End-to-End Transparency

This notebook documents the local code path we used to go from raw paired-end FASTQs to the WES ClinCNV runs:

- `/mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_baf_bedcoverage_wes`
- `/mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_no_baf_bedcoverage_wes`

It is meant for transparency and handoff. The emphasis is on showing the real scripts, paths, and parameter choices rather than wrapping everything in a new framework.

## Bundle Layout

- `01_fastq_to_bam/`: FASTQ to deduplicated BAMs
- `02_somatic_vcf/`: paired Mutect2 and post-filtering
- `03_baf_and_coverage/`: BAF creation, off-target BED generation, and BedCoverage matrices
- `04_clincnv_runs/`: final ClinCNV run scripts

The companion file `MANIFEST.tsv` maps each bundled file back to its original location.

## 1. FASTQs to Deduplicated BAMs

The BAM-generation stage is represented here by a GATK-style Snakemake pipeline and wrappers that prepare sample-level configs from a FASTQ metadata table.

Primary files:

- `01_fastq_to_bam/snakemake_wf.py`
- `01_fastq_to_bam/config.yaml`
- `01_fastq_to_bam/run_all.py`
- `01_fastq_to_bam/double_csv.py`
- `01_fastq_to_bam/double_fastq.py`

Core preprocessing steps in `snakemake_wf.py`:

1. `FastqToSam`
2. `RevertSam`
3. `MarkIlluminaAdapters`
4. `SamToFastq | bwa mem | MergeBamAlignment`
5. `MarkDuplicates`

The end product of this stage is a deduplicated BAM plus BAI for each sample. Those BAMs are the inputs for the somatic tumor-normal calling stage.

In [ ]:
fastq_to_bam_files = [
    "01_fastq_to_bam/snakemake_wf.py",
    "01_fastq_to_bam/config.yaml",
    "01_fastq_to_bam/run_all.py",
    "01_fastq_to_bam/double_csv.py",
    "01_fastq_to_bam/double_fastq.py",
]
fastq_to_bam_files

### BAM-generation Notes

- `run_all.py` reads `../bam_generation_v3/All4FastQSamples_fix.csv` and generates per-sample Snakemake configs.
- It points to the GRCh38 analysis-set reference under `/home/ubuntu/reference`.
- Output BAMs are uploaded to `s3://rocken-matched-melanoma-panel-seq/analysis/bam_generation_v3`.
- The somatic workflow later expects deduplicated BAMs and their indexes from that location.

## 2. Tumor-Normal BAMs to Somatic VCFs

The paired somatic calling stage is captured by:

- `02_somatic_vcf/vcf_gen_v3.sh`
- `02_somatic_vcf/filter_vcfs.sh`

`vcf_gen_v3.sh` downloads tumor and normal BAMs for each pair, runs `Mutect2`, writes an unfiltered VCF plus the F1R2 tarball, and uploads the results.

`filter_vcfs.sh` then performs the standard downstream filtering pieces:

1. `LearnReadOrientationModel`
2. `GetPileupSummaries` for tumor and normal
3. `CalculateContamination`
4. `FilterMutectCalls`

That yields a filtered VCF for each tumor-normal pair.

In [ ]:
somatic_vcf_files = [
    "02_somatic_vcf/vcf_gen_v3.sh",
    "02_somatic_vcf/filter_vcfs.sh",
]
somatic_vcf_files

### Somatic-calling Notes

- `vcf_gen_v3.sh` expects a local `pairs.txt` with `tumor,normal` sample IDs.
- It uses the same GRCh38 analysis-set reference and a panel BED with 100 bp padding.
- `filter_vcfs.sh` reads the pair folders back from the VCF S3 prefix and emits filtered VCFs to a `post_mutect2` subfolder.
- For the ClinCNV BAF-generation step below, the script we used reads the pair VCFs and extracts per-sample BAF values from those VCF records.

## 3. Pair VCFs to BAF Folder

The cleaned BAF folder was created with:

- `03_baf_and_coverage/baf_from_pair_vcfs.sh`

This script reads one pair VCF per tumor-normal pair, keeps biallelic SNVs with non-missing genotype, allele depths, and sufficient depth, and writes one TSV per sample.

Important details implemented in the script:

- VCF input root defaults to `/mnt/myvolume/panel_seq/new_bed_analysis/vcfs`
- Pair list defaults to `/mnt/myvolume/panel_seq/new_bed_analysis/pairs_df_filtered.csv`
- Output defaults to `/mnt/myvolume/panel_seq/new_bed_analysis/baf_from_pair_vcfs_clean`
- `DP >= 10`
- Output columns are `chr start end chr_pos baf depth`
- If a sample appears across multiple pairs, duplicate positions are resolved by keeping the highest-depth row

In [ ]:
baf_command = r'''bash /mnt/myvolume/panel_seq/new_bed_analysis/baf_from_pair_vcfs.sh'''
print(baf_command)

## 4. BAMs to BedCoverage Matrices for WES

The WES ClinCNV run uses BedCoverage-derived on-target and off-target matrices.

Files in this bundle:

- `03_baf_and_coverage/gen_wes_offtarget_bed_100kb.sh`
- `03_baf_and_coverage/build_wes_bedcoverage_matrices.sh`
- `03_baf_and_coverage/bedcoverage_wes_full_transparency.ipynb`

The off-target BED script does the WES-specific binning described for the ClinCNV setup:

- start from the genome minus target regions
- bin off-target space into 100 kb windows
- drop bins smaller than 50 kb
- compute GC values per bin

The BedCoverage matrix script then:

- downloads each sample BAM from the BAM S3 prefix
- runs `BedCoverage` on the target BED with `min_mapq 0`
- runs `BedCoverage` on the WES off-target BED with `min_mapq 10`
- merges the per-sample coverage outputs into matrix-style `.cov` files

In [ ]:
coverage_commands = [
    "bash /mnt/myvolume/panel_seq/new_bed_analysis/gen_wes_offtarget_bed_100kb.sh",
    "bash /mnt/myvolume/panel_seq/new_bed_analysis/build_wes_bedcoverage_matrices.sh",
]
coverage_commands

### Coverage Files Used by the Final WES Runs

- `/mnt/myvolume/panel_seq/new_bed_analysis/normal.ontarget.wes_mapq0_bedcoverage.cov`
- `/mnt/myvolume/panel_seq/new_bed_analysis/tumor.ontarget.wes_mapq0_bedcoverage.cov`
- `/mnt/myvolume/panel_seq/new_bed_analysis/normal.offtarget.wes_100kb_mapq10_bedcoverage.cov`
- `/mnt/myvolume/panel_seq/new_bed_analysis/tumor.offtarget.wes_100kb_mapq10_bedcoverage.cov`
- `/mnt/myvolume/panel_seq/new_bed_analysis/ssSC_v5.gc.genes.bed`
- `/mnt/myvolume/panel_seq/new_bed_analysis/clincnv_offtarget_wes_100kb_filtered.bed`

## 5. BedCoverage Matrices to ClinCNV Calls

The final WES ClinCNV runs are represented by two scripts:

- `04_clincnv_runs/run_clincnv_baf_bedcoverage_wes.sh`
- `04_clincnv_runs/run_clincnv_no_baf_bedcoverage_wes.sh`

Both use:

- `--colNum 4`
- `--lengthS 9`
- `--filterStep 2`
- `--scoreS 200`
- WES on-target and off-target BedCoverage matrices

The only branch point is whether the BAF folder is passed.

In [ ]:
clincnv_commands = {
    "with_baf": "bash /mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_baf_bedcoverage_wes/run_clincnv.sh",
    "without_baf": "bash /mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_no_baf_bedcoverage_wes/run_clincnv.sh",
}
clincnv_commands

### Exact ClinCNV Branching

With BAF:

- output: `/mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_baf_bedcoverage_wes`
- includes `--bafFolder /mnt/myvolume/panel_seq/new_bed_analysis/baf_from_pair_vcfs_clean`

Without BAF:

- output: `/mnt/myvolume/panel_seq/new_bed_analysis/new_clincnv_runs/full_wes_s200_l9_f2_no_baf_bedcoverage_wes`
- same WES BedCoverage inputs, but no `--bafFolder`

Both call the same ClinCNV R script:

- `/mnt/myvolume/panel_seq/reset_analysis_sample/test_2/ClinCNV/clinCNV.R`

## End-to-End Summary

The practical flow is:

1. FASTQs are aligned and deduplicated into BAM/BAI files.
2. Tumor-normal BAM pairs are called with `Mutect2`.
3. Mutect2 outputs are filtered with `FilterMutectCalls`-related steps.
4. Pair VCFs are transformed into a per-sample BAF folder.
5. Sample BAMs are profiled with `BedCoverage` into WES on-target and off-target matrices.
6. ClinCNV is run either with the BAF folder or without it.

This bundle exists so that each of those transitions is visible in concrete local code.